In [ ]:
import os
print(os.getcwd())
from dotenv import load_dotenv
import os

load_dotenv()
hf_token = os.getenv("HF_TOKEN")
print(hf_token)

In [ ]:
from huggingface_hub import model_info, login
import os

hf_token = os.getenv("HF_TOKEN")
login(token=hf_token)

try:
    # This will fail if token doesn't have access to Virchow
    info = model_info("paige-ai/Virchow", token=hf_token)
    print(f"✓ Token is valid! Model info: {info.id}")
except Exception as e:
    print(f"✗ Token error: {e}")

In [ ]:
# =============================================================================
# Corrected Prism Feature Extraction Pipeline
# =============================================================================
# KEY FIXES vs original code:
#   1. Uses Virchow V1 (paige-ai/Virchow) — Prism was trained with V1, NOT V2
#   2. FiveCrop crops are averaged into ONE 2560-d vector per patch
#   3. All patch vectors are stacked into (1, N_patches, 2560) per slide
#   4. Prism slide_representations() is called ONCE per slide, not per patch
# =============================================================================
 
# ─── Code Cell 1: Pipeline Class ─────────────────────────────────────────────
 
import os
import glob
import torch
import timm
from PIL import Image
from typing import Union, List
from torchvision import transforms
from timm.data import resolve_data_config
from timm.layers import SwiGLUPacked
from transformers import AutoModel
from huggingface_hub import login
 
 
class PrismFullPipeline:
    """
    Corrected Prism workflow:
 
    Step 1 — Tile encoding (Virchow V1):
        Each image patch  →  FiveCrop  →  5 embeddings  →  mean  →  1 × 2560-d vector
 
    Step 2 — Slide aggregation (Prism Perceiver):
        All N patch vectors  →  stack (1, N, 2560)  →  slide_representations()
        →  slide embedding (1, 1280)
 
    IMPORTANT: slide_representations() must be called ONCE per whole slide,
    not once per patch.
    """
 
    def __init__(self, hf_token: str, device: str = None):
        login(token=hf_token)
        self.device = torch.device(
            device if device else ("cuda" if torch.cuda.is_available() else "cpu")
        )
 
        # ── Prism slide-level model ───────────────────────────────────────────
        print("[Prism] Loading slide-level model (paige-ai/Prism)...")
        self.prism_model = AutoModel.from_pretrained(
            "paige-ai/Prism",
            trust_remote_code=True,   # required
        ).to(self.device).eval()
 
        # ── Virchow V1 tile encoder ───────────────────────────────────────────
        # FIX #1: Use Virchow V1, NOT Virchow2.
        # Prism's Perceiver was pretrained on Virchow V1 embeddings (2560-d).
        # Virchow2 produces embeddings from a different distribution and will
        # silently degrade Prism's performance.
        print("[Virchow] Loading tile-level encoder (paige-ai/Virchow) — V1 ...")
        self.tile_encoder = timm.create_model(
            "hf-hub:paige-ai/Virchow",
            pretrained=True,
            mlp_layer=SwiGLUPacked,      # ← this was missing
            act_layer=torch.nn.SiLU,     # ← this too
            num_classes=0,
        ).to(self.device).eval()
 
        # Resolve normalisation stats from the model's own config
        cfg = resolve_data_config(self.tile_encoder.pretrained_cfg, model=self.tile_encoder)
        self._base_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=cfg["mean"], std=cfg["std"]),
        ])
 
        # FiveCrop: crops to 256 px, then resizes each crop to 224 for Virchow
        # The outer image should therefore be at least 256 × 256 px.
        self._fivecrop_transform = transforms.Compose([
            transforms.FiveCrop(256),
            transforms.Lambda(
                lambda crops: torch.stack([self._base_transform(c) for c in crops])
            ),
        ])
 
    # ─────────────────────────────────────────────────────────────────────────
    # Step 1 — encode ONE patch image → ONE 2560-d vector
    # ─────────────────────────────────────────────────────────────────────────
    @torch.inference_mode()
    def encode_patch(self, image: Union[str, Image.Image]) -> torch.Tensor:
        """
        Encode a single patch image into a 2560-d tile embedding.
 
        FIX #2 & #3:
          - Extract features for all 5 crops (shape 5 × tokens × 1280).
          - Combine CLS + mean-of-patches  →  2560-d per crop.
          - Average the 5 crops into ONE vector per patch (shape 2560,).
            This is the unit that Prism expects in the tile sequence.
 
        Returns:
            Tensor of shape (2560,)  — one embedding per patch.
        """
        if isinstance(image, str):
            image = Image.open(image).convert("RGB")
 
        crops = self._fivecrop_transform(image).to(self.device)  # (5, 3, 224, 224)
 
        with torch.autocast(device_type=self.device.type, dtype=torch.float16):
            features = self.tile_encoder.forward_features(crops)
            # features: (5, num_tokens, 1280)
            # Token layout: [CLS, reg1, reg2, reg3, reg4, patch_1, patch_2, ...]
            cls_token  = features[:, 0]        # (5, 1280)
            patch_mean = features[:, 5:].mean(dim=1)  # (5, 1280)  skip register tokens
            combined   = torch.cat([cls_token, patch_mean], dim=-1)  # (5, 2560)
 
        # Average across the 5 crops → single 2560-d vector for this patch
        return combined.mean(dim=0)  # (2560,)
 
    # ─────────────────────────────────────────────────────────────────────────
    # Step 2 — aggregate patch embeddings and get slide representation
    # ─────────────────────────────────────────────────────────────────────────
    @torch.inference_mode()
    def get_slide_embedding(self, patch_embeddings: torch.Tensor) -> dict:
        """
        Run Prism's Perceiver on the full tile sequence for ONE slide.
 
        FIX #4: This must be called ONCE per slide with ALL patch embeddings
        already stacked, not once per individual patch.
 
        Args:
            patch_embeddings: Tensor of shape (N_patches, 2560)
                              where N_patches is every valid patch in the slide.
 
        Returns:
            dict with keys:
                'image_embedding'  — (1, 1280)  global slide representation
                'image_latents'    — (1, 512, 1280)  latent features for generation
        """
        # Add batch dimension: (1, N_patches, 2560)
        tile_embeddings = patch_embeddings.unsqueeze(0).to(self.device)
 
        with torch.autocast(device_type=self.device.type, dtype=torch.float16):
            reprs = self.prism_model.slide_representations(tile_embeddings)
 
        return reprs
print(hf_token)

In [ ]:

# ─── Code Cell 2: Per-patch feature extraction (save .pt per patch) ──────────
 
import pandas as pd
from tqdm import tqdm
 
 
def extract_patch_features_for_slide(
    patch_paths: List[str],
    pipeline: PrismFullPipeline,
    output_dir: str,
    skip_existing: bool = True,
) -> None:
    """
    Encode every patch in `patch_paths` and save each as a (2560,) .pt file.
 
    These files are INTERMEDIATE outputs — one 2560-d vector per patch.
    They are later loaded and stacked to form the slide-level input to Prism.
 
    Args:
        patch_paths:   List of PNG file paths belonging to ONE slide.
        pipeline:      PrismFullPipeline instance.
        output_dir:    Directory where <patch_stem>.pt files are saved.
        skip_existing: Skip patches that already have a saved .pt file.
    """
    os.makedirs(output_dir, exist_ok=True)
 
    if skip_existing:
        done = {os.path.splitext(f.name)[0] for f in os.scandir(output_dir) if f.name.endswith(".pt")}
        patch_paths = [p for p in patch_paths if os.path.splitext(os.path.basename(p))[0] not in done]
 
    for patch_path in tqdm(patch_paths, desc="  Encoding patches", leave=False, unit="patch"):
        stem = os.path.splitext(os.path.basename(patch_path))[0]
        save_path = os.path.join(output_dir, stem + ".pt")
        try:
            embedding = pipeline.encode_patch(patch_path)  # (2560,)
            torch.save(embedding.cpu(), save_path)
        except Exception as e:
            tqdm.write(f"  ❌ Failed on {patch_path}: {e}")
 

In [ ]:

# ─── Code Cell 3: Slide-level aggregation ────────────────────────────────────
 
def compute_slide_embedding(
    patch_feat_dir: str,
    pipeline: PrismFullPipeline,
) -> torch.Tensor:
    """
    Load all per-patch .pt files for one slide, stack them, and run Prism.
 
    Returns:
        slide_embedding — Tensor of shape (1, 1280).
    """
    pt_files = sorted(glob.glob(os.path.join(patch_feat_dir, "*.pt")))
    if not pt_files:
        raise FileNotFoundError(f"No .pt files found in {patch_feat_dir}")
 
    # Stack all patch embeddings: (N_patches, 2560)
    patch_embeddings = torch.stack([torch.load(p, map_location="cpu") for p in pt_files])
    print(f"  Loaded {len(pt_files)} patch embeddings → shape {patch_embeddings.shape}")
 
    reprs = pipeline.get_slide_embedding(patch_embeddings)
    return reprs["image_embedding"]  # (1, 1280)
 
 

In [ ]:

# ─── Code Cell 4: Full pipeline — all slides ─────────────────────────────────
 
def run_full_pipeline(
    patch_data_dir: str,
    pipeline: PrismFullPipeline,
    non_white_csv: str,
    patch_feat_root: str,
    slide_embed_root: str,
) -> None:
    """
    Two-stage pipeline for all slides:
 
    Stage 1 — Patch encoding (Virchow V1):
        For each slide:
            for each valid (non-white) patch:
                encode_patch() → save (2560,) .pt file
 
    Stage 2 — Slide embedding (Prism Perceiver):
        For each slide:
            load all .pt patch embeddings
            stack → (N_patches, 2560)
            get_slide_embedding() → save (1, 1280) .pt file
 
    Args:
        patch_data_dir:   Root dir containing one sub-folder per slide with .png patches.
        pipeline:         PrismFullPipeline instance.
        non_white_csv:    CSV with column 'patch_name' listing valid (non-background) patches.
        patch_feat_root:  Where per-patch (2560,) .pt files are saved.
        slide_embed_root: Where per-slide (1, 1280) .pt files are saved.
    """
    os.makedirs(patch_feat_root, exist_ok=True)
    os.makedirs(slide_embed_root, exist_ok=True)
 
    # Load valid patch filter
    non_white_df = pd.read_csv(non_white_csv)
    valid_names = set(non_white_df["patch_name"].astype(str))
    print(f"📊 {len(valid_names)} valid patch names loaded from CSV")
 
    slide_dirs = sorted(d for d in glob.glob(os.path.join(patch_data_dir, "*")) if os.path.isdir(d))
    print(f"🧩 Processing {len(slide_dirs)} slides...\n")
 
    for slide_dir in tqdm(slide_dirs, desc="Slides", unit="slide"):
        slide_name = os.path.basename(slide_dir)
        patch_feat_dir = os.path.join(patch_feat_root, slide_name)
        slide_embed_path = os.path.join(slide_embed_root, slide_name + ".pt")
 
        # ── Stage 1: encode patches ──────────────────────────────────────────
        all_patches = glob.glob(os.path.join(slide_dir, "*.png"))
        valid_patches = [p for p in all_patches if os.path.basename(p) in valid_names]
 
        if not valid_patches:
            tqdm.write(f"  ⚠️  {slide_name}: no valid patches, skipping")
            continue
 
        tqdm.write(f"  ▶  {slide_name}: {len(valid_patches)} valid patches")
        extract_patch_features_for_slide(valid_patches, pipeline, patch_feat_dir)
 
        # ── Stage 2: aggregate → slide embedding ────────────────────────────
        if os.path.exists(slide_embed_path):
            tqdm.write(f"  ⏭️  {slide_name}: slide embedding already exists, skipping Stage 2")
            continue
 
        try:
            slide_emb = compute_slide_embedding(patch_feat_dir, pipeline)
            torch.save(slide_emb.cpu(), slide_embed_path)
            tqdm.write(f"  ✅  {slide_name}: slide embedding saved → {slide_emb.shape}")
        except Exception as e:
            tqdm.write(f"  ❌  {slide_name}: Stage 2 failed — {e}")
 
 

In [ ]:

# ─── Code Cell 5: Entry point ────────────────────────────────────────────────
 # ─── Code Cell 5: Entry point ────────────────────────────────────────────

if __name__ == "__main__":
    # 1. Authenticate FIRST with explicit login
    from huggingface_hub import login
    
    hf_token = os.getenv("HF_TOKEN")
    if not hf_token:
        raise ValueError("HF_TOKEN not found in .env file!")
    
    print(f"Authenticating with HuggingFace token...")
    login(token=hf_token, add_to_git_credential=False)
    
    # 2. Now instantiate pipeline
    pipeline = PrismFullPipeline(hf_token=hf_token)
    
    # ... rest of code
    # 2. Run full pipeline
    run_full_pipeline(
        patch_data_dir   = r"D:\Aamir Gulzar\KSA_project2\dataset\patch_data",
        pipeline         = pipeline,
        non_white_csv    = r"D:\Aamir Gulzar\KSA_project2\dataset\final_merged_without_white.csv",
        patch_feat_root  = r"D:\Aamir Gulzar\KSA_project2\dataset\Features\Prism_patches",
        slide_embed_root = r"D:\Aamir Gulzar\KSA_project2\dataset\Features\Prism_slides",
    )
 

In [ ]:

# ─── Code Cell 5: Entry point ────────────────────────────────────────────────
 # ─── Code Cell 5: Entry point ────────────────────────────────────────────

if __name__ == "__main__":
    # 1. Authenticate FIRST with explicit login
    from huggingface_hub import login
    
    hf_token = os.getenv("HF_TOKEN")
    if not hf_token:
        raise ValueError("HF_TOKEN not found in .env file!")
    
    print(f"Authenticating with HuggingFace token...")
    login(token=hf_token, add_to_git_credential=False)
    
    # 2. Now instantiate pipeline
    pipeline = PrismFullPipeline(hf_token=hf_token)
    
    # ... rest of code
    # 2. Run full pipeline
    run_full_pipeline(
        patch_data_dir   = r"D:\Aamir Gulzar\KSA_project2\dataset\patch_data",
        pipeline         = pipeline,
        non_white_csv    = r"D:\Aamir Gulzar\KSA_project2\dataset\final_merged_without_white.csv",
        patch_feat_root  = r"D:\Aamir Gulzar\KSA_project2\dataset\Features\Prism_patches",
        slide_embed_root = r"D:\Aamir Gulzar\KSA_project2\dataset\Features\Prism_slides",
    )
 

In [ ]:
# PRISM (1, 1280) Feature Validator - Clean Version

import torch
import random
from pathlib import Path
from tqdm import tqdm
import pandas as pd
from IPython.display import display

# ─── CONFIG ───────────────────────────────────────────────
AGG_DIR     = r"D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\PRISM"
EXPECTED_SHAPE = (1, 1280)
SAMPLE_RATE = 1.0   # 1.0 = check all files
# ─────────────────────────────────────────────────────────


def validate_file(path: Path) -> dict:
    result = {
        "file": path.name,
        "ok": False,
        "error": None,
        "shape": None,
        "dtype": None,
        "file_size_kb": round(path.stat().st_size / 1024, 2),
        "has_nan": False,
        "has_inf": False,
        "is_all_zero": False,
        "min": None,
        "max": None,
        "mean": None,
        "std": None,
    }

    try:
        tensor = torch.load(path, map_location="cpu")

        # ── Must be tensor ───────────────────────────────
        if not isinstance(tensor, torch.Tensor):
            result["error"] = f"Not a tensor: {type(tensor)}"
            return result

        result["shape"] = tuple(tensor.shape)
        result["dtype"] = str(tensor.dtype)

        # ── Shape check ─────────────────────────────────
        if result["shape"] != EXPECTED_SHAPE:
            result["error"] = f"Wrong shape {result['shape']}, expected {EXPECTED_SHAPE}"
            return result

        if tensor.numel() == 0:
            result["error"] = "Empty tensor"
            return result

        # Flatten (1,1280) → (1280,)
        t = tensor.float().view(-1)

        # ── NaN / Inf check ─────────────────────────────
        result["has_nan"] = bool(torch.isnan(t).any())
        result["has_inf"] = bool(torch.isinf(t).any())

        if result["has_nan"]:
            result["error"] = "Contains NaN"
            return result

        if result["has_inf"]:
            result["error"] = "Contains Inf"
            return result

        # ── Zero vector check ───────────────────────────
        result["is_all_zero"] = bool(torch.all(t == 0))

        # ── Stats ───────────────────────────────────────
        result["min"]  = round(float(t.min()), 6)
        result["max"]  = round(float(t.max()), 6)
        result["mean"] = round(float(t.mean()), 6)
        result["std"]  = round(float(t.std()), 6)

        result["ok"] = True

    except Exception as e:
        result["error"] = str(e)

    return result


def run_check(agg_dir: str, sample_rate: float = SAMPLE_RATE):
    root = Path(agg_dir)
    files = list(root.glob("*.pt"))

    print(f"📂 Directory: {root}")
    print(f"📄 Files found: {len(files):,}\n")

    if not files:
        print("❌ No .pt files found")
        return None

    # Sampling
    if sample_rate < 1.0:
        n = max(1, int(len(files) * sample_rate))
        files = random.sample(files, n)
        print(f"Sampling {n} files\n")
    else:
        print(f"Checking all files\n")

    results = []
    for f in tqdm(files, desc="Validating"):
        results.append(validate_file(f))

    df = pd.DataFrame(results)

    ok = df[df["ok"] == True]
    bad = df[df["ok"] == False]

    # ── REPORT ─────────────────────────────────────────
    print("\n" + "="*60)
    print("VALIDATION REPORT")
    print("="*60)
    print(f"Total checked : {len(df)}")
    print(f"✓ Valid       : {len(ok)}")
    print(f"✗ Invalid     : {len(bad)}")

    # Shape distribution
    print("\nShape Distribution:")
    display(df["shape"].value_counts())

    # Dtype distribution
    print("\nDType Distribution:")
    display(df["dtype"].value_counts())

    # File size
    print("\nFile Size Stats (KB) — expected ~5 KB:")
    display(df["file_size_kb"].describe().round(2))

    # Stats
    if len(ok) > 0:
        print("\nFeature Value Stats:")
        display(ok[["min", "max", "mean", "std"]].describe().round(6))

        print(f"\nAll-zero vectors: {ok['is_all_zero'].sum()}")

    # Errors
    if len(bad) > 0:
        print("\nError Summary:")
        display(bad["error"].value_counts())

        print("\nBad Files:")
        display(bad[["file", "shape", "error", "file_size_kb"]])

        bad.to_csv(root / "corrupt_files.csv", index=False)
        print("\n💾 Saved corrupt_files.csv")

    else:
        print("\n✅ All files are valid!")

    return df


# ─── RUN ───────────────────────────────────────────────
df_results = run_check(AGG_DIR, SAMPLE_RATE)